In [ ]:
import matplotlib.pyplot as plt
import torch

In [ ]:
def rotate_vec(x, rads):
    """
    returns rotated x by rads radians
    """
    if type(rads) is not torch.Tensor:
        rads = torch.tensor(rads)

    cos_r = torch.cos(rads)
    sin_r = torch.sin(rads)
    rotation_matrix = torch.tensor([[cos_r, -sin_r], [sin_r, cos_r]])
    return rotation_matrix @ x


def noise_vec(x, pct=0.3):
    """
    returns x+v, where v is a randn vector with magnitude pct*|x|
    """
    v = torch.randn_like(x)
    v /= torch.norm(v, dim=-1, keepdim=True)
    v *= torch.norm(x, dim=-1, keepdim=True) * pct

    return x + v

In [ ]:
d_kv = 8
d = 16

base_1 = torch.randn(d_kv)
base_2 = torch.randn(d_kv)
rotations = 2
n_heads = (1 + rotations) * 2

full = torch.stack(
    [base_1] + [noise_vec(base_1) for r in range(rotations)] + [base_2] + [noise_vec(base_2) for r in range(rotations)]
)
o = torch.nn.Linear(n_heads * d_kv, d, bias=False)

full.shape, o.weight.shape

In [ ]:
gramian = (full @ full.T) / (full.norm(dim=-1, keepdim=True) @ full.norm(dim=-1, keepdim=True).T)
plt.imshow(gramian, cmap="RdBu", vmin=-1, vmax=1, origin="upper")
plt.colorbar()

In [ ]:
# similarity threshold
epsilon = 0.9999

# each i,j element represents sim(h_i, h_j)
similarities = gramian
num_heads = similarities.shape[0]

# magnitudes, represents mags of each vector
mags = full.norm(dim=-1)

groups = []
pivots = []

# sorted in order of largest to smallest heads in magnitude
_, sorted_heads = torch.topk(mags, num_heads)
sorted_heads = [head.item() for head in sorted_heads]

In [ ]:
o.weight.shape

In [ ]:
# group the heads
while len(sorted_heads) != 0:
    max_head = sorted_heads.pop(0)

    # find all heads h_i such that sim(max_head, h_i) > epsilon
    similar_heads = [max_head]
    removal = []
    for head in sorted_heads:
        if torch.abs(similarities[max_head, head]) > epsilon:
            removal.append(head)
            similar_heads.append(head)

    for head in removal:
        sorted_heads.remove(head)

    groups.append(similar_heads)

# determine which heads to retain and the new o
pivot_heads = {}
# new_out = []
for group in groups:
    # find pivot head
    max_score = 0
    pivot_head = group[0]
    for head in group:
        score = sum(abs(similarities[head, j]) for j in group if j != head)
        if score > max_score:
            max_score = score
            pivot_head = head

    pivot_heads[pivot_head] = group

    # assemble new row of o
    # pivot_mag = mags[pivot_head]
    # pivot_col = o.weight[:, pivot_head*d_kv:(pivot_head+1)*d_kv]
    # for head in group:
    #     if head == pivot_head:
    #         continue

    #     pivot_col = pivot_col + o.weight[:, head*d_kv:(head+1)*d_kv] * mags[head]/pivot_mag

    # new_out.append(pivot_col)

# new_out = torch.cat(new_out, dim=-1)

head_multiples = []
for p in pivot_heads:
    p_mag = mags[p]
    group = pivot_heads[p]
    multiplier = 0
    for h in group:
        multiplier += (mags[h] / p_mag) * torch.sign(similarities[p, h])

    head_multiples.append((p, multiplier.item()))

In [ ]:
head_multiples